#### 10.1
Risk Parity, Normal Assumption using test5_2.csv and testout10_1.csv

In [19]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [20]:
data = pd.read_csv('../testfiles/data/test5_2.csv').values
data

array([[0.08497905, 0.11678108, 0.04230441, 0.00898354, 0.00387595],
       [0.11678108, 0.16048451, 0.05813615, 0.01234549, 0.00532646],
       [0.04230441, 0.05813615, 0.03744009, 0.00596294, 0.00257271],
       [0.00898354, 0.01234549, 0.00596294, 0.00168834, 0.00054633],
       [0.00387595, 0.00532646, 0.00257271, 0.00054633, 0.00031428]])

In [21]:
def risk_parity_objective(w, cov):
    port_vol = np.sqrt(np.dot(w, np.dot(cov, w)))
    marginal_risk = np.dot(cov, w)

    risk_comp = w * marginal_risk / port_vol
    
    avg_risk = np.mean(risk_comp)
    deviations = risk_comp - avg_risk
    objective = np.sum(deviations ** 2) * 1e5
    
    return objective

In [22]:
def risk_parity_weights(cov):
    # Setup optimization
    num_assets = cov.shape[0]
    init_w = np.ones(num_assets) / num_assets
    
    constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
    weight_bounds = tuple((0, None) for _ in range(num_assets))
    
    # Run optimization
    optimization_result = minimize(
        risk_parity_objective,
        init_w,
        args=(cov,),
        method='SLSQP',
        bounds=weight_bounds,
        constraints=constraint,
        options={'ftol': 1e-9, 'maxiter': 1000}
    )
    
    # Extract and normalize weights
    final_weights = optimization_result.x
    final_weights = final_weights / np.sum(final_weights)
    
    for weight in final_weights:
        print(weight)

risk_parity_weights(data)

0.035463715082790316
0.025806189313222514
0.05646929008305454
0.2659201588453212
0.6163406466756115


##### 10.2
Risk Parity, Normal Assumption, 1/2 risk weight on X5, using test5_2.csv and testout10_2.csv

In [23]:
def risk_budget_objective(w, cov, budget_vec):
    port_vol = np.sqrt(np.dot(w, np.dot(cov, w)))
    marginal_risk = np.dot(cov, w)

    risk_contrib = w * marginal_risk / port_vol
    
    normal_contrib = risk_contrib / budget_vec
    avg_risk = np.mean(normal_contrib)
    deviations = normal_contrib - avg_risk
    objective = np.sum(deviations ** 2) * 1e5
    
    return objective

In [24]:
def risk_budget_weights(cov):
    # Setup optimization
    num_assets = cov.shape[0]
    init_w = np.ones(num_assets) / num_assets
    budget_allocation = np.array([1.0, 1.0, 1.0, 1.0, 0.5])
    
    constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
    weight_bounds = tuple((0, None) for _ in range(num_assets))
    
    # Run optimization
    result = minimize(
        risk_budget_objective,
        init_w,
        args=(cov, budget_allocation,),
        method='SLSQP',
        bounds=weight_bounds,
        constraints=constraint,
        options={'ftol': 1e-9, 'maxiter': 1000}
    )
    
    # Extract and normalize weights
    final_weights = result.x
    final_weights = final_weights / np.sum(final_weights)
    
    for weight in final_weights:
        print(weight)

risk_budget_weights(data)

0.05024249019597931
0.036560257326153756
0.08039926069295925
0.3786081734691065
0.45418981831580124


#### 10.3
Max Sharpe Ratio, normal assumption, w>0 using test5_2.csv, test10_3_means.csv, risk free rate=0.04 and testout10_3.csv

In [25]:
data2 = pd.read_csv('../testfiles/data/test10_3_means.csv').values
data2

array([[0.09],
       [0.08],
       [0.07],
       [0.06],
       [0.05]])

In [26]:
def negative_sharpe(w, mu, cov, rf):
    ret = w @ mu
    vol = np.sqrt(w @ cov @ w)
    return -(ret - rf) / vol

In [27]:
def max_sharpe_pos_w(mu, cov, rf):
    n = len(mu)
    w0 = np.ones(n) / n

    bounds = [(0, 1)] * n  # w > 0
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    result = minimize(
        negative_sharpe,
        w0,
        args=(mu, cov, rf),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    final_weights = result.x

    for weight in final_weights:
        print(weight)

max_sharpe_pos_w(data2, data, 0.04)

0.0
0.0
0.0
0.12174031375873032
0.8782598410381497


#### 10.4
Max Sharpe Ratio, normal assumption, 0.1 <= w <= 0.5, using test5_2.csv, test10_3_means.csv, rf=0.04 and testout10_4.csv

In [28]:
def max_sharpe_bounded(mu, cov, rf):
    n = len(mu)
    w0 = np.ones(n) / n

    bounds = [(0.1, 0.5)] * n  # 0.1 <= w <= 0.5
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    result = minimize(
        negative_sharpe,
        w0,
        args=(mu, cov, rf),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 1000}
    )

    final_weights = result.x

    for weight in final_weights:
        print(weight)

max_sharpe_bounded(data2, data, 0.04)

0.1
0.1
0.1
0.5
0.20000000000006182
